# Silver: TPC-H Logistics

The pipeline retains all eight entities and all 61 mapped attributes. It validates lossless casts, required values, primary keys, every foreign key (including the part/supplier pair), shipment chronology, categorical lists, and non-empty orders. Invalid rows are not silently dropped: any failed rule blocks publication of every Silver entity.

Each run appends checks with a run ID to `workspace.tpch_silver.quality_validation_results`. Failed runs append typed diagnostics to `quarantine_<table>` and raw cast diagnostics to `quarantine_<table>_casts` in the same schema. A row can appear once for each rule it violates. Correct the source issue with the Bronze owner and rerun; do not manually discard parent records.

Publication enforces the exact target types and required-column `NOT NULL` constraints. Existing schema drift causes an error rather than an automatic migration. The eight Delta overwrites are **not atomic together**: after a write failure, keep readers paused and rerun the entire notebook against the same verified, idle Bronze refresh. Only the final completion message means all eight writes succeeded.

In [0]:
import importlib
import logging
import os
import sys
from pathlib import Path
from uuid import uuid4

os.environ["DB_CATALOG"] = "workspace"
os.environ["SCHEMA_PREFIX"] = "tpch"

repo_root: Path = Path.cwd().parent
sys.path.insert(0, str(repo_root))

from config import env

importlib.reload(env)

from src.silver.quality import (
    load_logistics_rules,
    record_validation_results,
    validate_silver_tables,
)
from src.silver.transform import (
    publish_silver_tables,
    run_silver_transformations,
)

logging.basicConfig(level=logging.INFO)
run_id: str = str(uuid4())
bronze_namespace: str = f"{env.CATALOG}.{env.BRONZE_SCHEMA}"
silver_namespace: str = f"{env.CATALOG}.{env.SILVER_SCHEMA}"
print(f"Silver run {run_id}: {bronze_namespace} -> {silver_namespace}")

rules = load_logistics_rules()
transformed = run_silver_transformations(spark, bronze_namespace)
row_counts: dict[str, int] = {
    table: dataframe.count() for table, dataframe in transformed.tables.items()
}

validation = validate_silver_tables(transformed, rules)
record_validation_results(spark, validation, run_id, silver_namespace)
display(validation.report.orderBy("table_name", "rule_name"))
validation.assert_valid()
publish_silver_tables(spark, transformed, validation, silver_namespace)

display(
    spark.createDataFrame(
        [(run_id, table, count) for table, count in row_counts.items()],
        "run_id STRING, table_name STRING, published_rows LONG",
    ).orderBy("table_name")
)
print(f"Silver run {run_id} completed: all eight tables published")